# Introduction to NumPy for Spatial Programming

## Why NumPy?
In the world of GIS, data usually comes in two flavors:
1.  **Rasters:** Grids of pixels (satellite imagery, elevation models).
2.  **Vectors:** Shapes with attributes (tables of data).

Under the hood, ArcGIS Pro uses **NumPy** (Numerical Python) to process this data.
* A **Raster** is just a NumPy 2D array.
* An **Attribute Table** can be treated as a NumPy structured array.

Understanding NumPy allows you to perform "Map Algebra" and complex statistical analysis much faster than standard Python loops.

In [1]:
import numpy as np
import os

# 1. Create a dummy ASCII Raster file (Elevation data)
# This simulates a small 5x5 Digital Elevation Model (DEM)
raster_header = "ncols 5\nnrows 5\nxllcorner 0\nyllcorner 0\ncellsize 10\nNODATA_value -9999\n"
raster_data = "100 102 105 110 108\n101 103 108 112 110\n105 109 115 120 118\n110 115 120 125 122\n112 118 122 128 -9999"

with open("dummy_dem.asc", "w") as f:
    f.write(raster_header + raster_data)

# 2. Create a dummy CSV file (Attribute data)
csv_header = "ParcelID,LandUse,Area_Sqm,Value_USD\n"
csv_data = "101,Residential,500.0,250000\n102,Commercial,1200.5,1500000\n103,Residential,450.2,210000\n104,Industrial,3000.0,800000\n105,Park,8000.0,0"

with open("dummy_attributes.csv", "w") as f:
    f.write(csv_header + csv_data)

print("Dummy GIS data files created successfully.")

Dummy GIS data files created successfully.


## Part 1: The Array Concept
Native Python uses `lists` to store data. Lists are flexible but slow for math.
NumPy uses `arrays`. Arrays are:
1.  **Fixed Type:** All items must be the same data type (e.g., all integers).
2.  **Vectorized:** You can do math on the whole array at once (no loops!).

In [2]:
# --- Standard Python List ---
elevations_list = [100, 102, 105, 110, 108]

# Task: Convert elevation from meters to feet (multiply by 3.28)
# We have to use a loop (slow and verbose)
elevations_ft_list = []
for elev in elevations_list:
    elevations_ft_list.append(elev * 3.28)

print(f"List calculation: {elevations_ft_list}")

# --- NumPy Array ---
# Create an array from the list
elevations_arr = np.array(elevations_list)

# Task: Convert to feet
# We multiply the whole object (Fast and clean - like Map Algebra)
elevations_ft_arr = elevations_arr * 3.28

print(f"Array calculation: {elevations_ft_arr}")

List calculation: [328.0, 334.56, 344.4, 360.79999999999995, 354.23999999999995]
Array calculation: [328.   334.56 344.4  360.8  354.24]


## Part 2: Working with Rasters (2D Arrays)
A raster is essentially a matrix. Let's load our dummy elevation file and treat it like a raster layer.

In [3]:
# We will skip the header rows to just get the data
# This creates a 2D array
dem_raster = np.loadtxt("dummy_dem.asc", skiprows=6)

print("--- Raster Properties ---")
print(f"Shape (Rows, Cols): {dem_raster.shape}")
print(f"Data Type: {dem_raster.dtype}")
print("\n--- The Data Matrix ---")
print(dem_raster)

# Note the -9999. In GIS, this is NoData.

--- Raster Properties ---
Shape (Rows, Cols): (5, 5)
Data Type: float64

--- The Data Matrix ---
[[  100.   102.   105.   110.   108.]
 [  101.   103.   108.   112.   110.]
 [  105.   109.   115.   120.   118.]
 [  110.   115.   120.   125.   122.]
 [  112.   118.   122.   128. -9999.]]


In [7]:
# 1. Handling NoData
# Let's mask out the -9999 value so it doesn't mess up our math
dem_masked = np.where(dem_raster == -9999, np.nan, dem_raster)

# 2. Simple Map Algebra: Calculate Slope Estimate
# (Simplified: Just the difference between a pixel and its neighbor to the right)
# We use slicing: array[row_start:row_end, col_start:col_end]
# This is not the same as neighborhood "Focal Statistics" or "Slope" tool logic, 
# but it is a simple way to calculate EW differences
left_pixels = dem_masked[:, :-1]  # All rows, all cols except last
right_pixels = dem_masked[:, 1:]  # All rows, all cols except first

# Calculate difference
slope_estimate = right_pixels - left_pixels

print("\n--- 'Slope' Estimate (Right - Left) ---")
print(slope_estimate)

# 3. Reclassify
# Find all areas higher than 115 meters
high_elevation = dem_masked > 115
print(f"\n--- Boolean Mask (Elev > 115) ---\n{high_elevation}")


--- Slope Estimate (Right - Left) ---
[[ 2.  3.  5. -2.]
 [ 2.  5.  4. -2.]
 [ 4.  6.  5. -2.]
 [ 5.  5.  5. -3.]
 [ 6.  4.  6. nan]]

--- Boolean Mask (Elev > 115) ---
[[False False False False False]
 [False False False False False]
 [False False False  True  True]
 [False False  True  True  True]
 [False  True  True  True False]]


## Part 3: Attribute Tables (Structured Arrays)
Standard arrays hold one data type (all floats or all ints).
But GIS attribute tables have columns with different types (ID is int, LandUse is text, Area is float).

NumPy handles this with **Structured Arrays**.

In [8]:
# 1. Define the Schema (The Data Type)
# 'U10' means Unicode string of max length 10
# 'i4' means 4-byte integer
# 'f4' means 4-byte float
gis_schema = np.dtype([
    ('ParcelID', 'i4'),
    ('LandUse', 'U15'),
    ('Area_Sqm', 'f4'),
    ('Value_USD', 'f4')
])

# 2. Load the CSV using this schema
# We skip the header row again
attributes = np.loadtxt("dummy_attributes.csv", skiprows=1, delimiter=",", dtype=gis_schema)

print("--- The Structured Array (Attribute Table) ---")
print(attributes)
# Notice it looks like a list of tuples, but it behaves like a table

--- The Structured Array (Attribute Table) ---
[(101, 'Residential',  500. ,  250000.)
 (102, 'Commercial', 1200.5, 1500000.)
 (103, 'Residential',  450.2,  210000.)
 (104, 'Industrial', 3000. ,  800000.) (105, 'Park', 8000. ,       0.)]


In [9]:
# 1. Accessing a Column (Field)
# Just like selecting a field in ArcPy or Pandas
areas = attributes['Area_Sqm']
print(f"\nAverage Parcel Area: {np.mean(areas)} Sqm")

# 2. Filtering (Definition Query)
# "Select By Attributes" where LandUse is 'Residential'
residential_parcels = attributes[attributes['LandUse'] == 'Residential']

print(f"\n--- Residential Parcels ---")
print(residential_parcels)

# 3. Calculation on Selection
# Calculate price per square meter only for Residential
price_per_sqm = residential_parcels['Value_USD'] / residential_parcels['Area_Sqm']

print(f"\nPrice per Sqm (Residential): {price_per_sqm}")


Average Parcel Area: 2630.14013671875 Sqm

--- Residential Parcels ---
[(101, 'Residential', 500. , 250000.) (103, 'Residential', 450.2, 210000.)]

Price per Sqm (Residential): [500.      466.45935]
